# Bit-Banging I2C with `scope.bitbanger`
`scope.bitbanger` can be made to speak many different protocols. In this example, we'll have it speak I2C.

Like 1-wire, I2C uses a bi-directional data line; however there is also a clock line and a power rail.

This notebook is written specifically for the **Microchip 24CS32 EEPROM**.

In this notebook, we'll start with showing the basics of using `scope.bitbanger` to talk I2C with a live target; these basics should be applicable to any I2C target.

We will then go beyond the basics to show what Husky can reveal about the 24CS32's internal behaviour; this will be very specific to the 24CS32, but you can use it as inspiration to explore this and other targets.

In order to run this notebook, you'll have to connect your own custom I2C target board to Husky; at this time we don't offer such pre-made target boards, but it's easy to make your own; here's the schematic for the one-off target we used to write this notebook:

<img src='img/circuit_i2c.png' width=400>

This circuit can easily be implemented on the blank target boards that come with our [starter kits](https://chipwhisperer.readthedocs.io/en/latest/Starter%20Kits/SCAPACK-L1.html), or on a breadboard that is connected to a [CW308](https://chipwhisperer.readthedocs.io/en/latest/Targets/CW308%20UFO.html) or [CW313](https://chipwhisperer.readthedocs.io/en/latest/Targets/CW313.html).

Consult your target's datasheet for recommended pullup resistor values. We've used `Rpup = 4.7` k$\ohm$ and `C = 30` nF. What matters most is that you get consistent successful communication with your target.

If you don't want to go through the trouble of making your own target but are still curious to see what this is all about, we've included screenshots of the more interesting power traces, so you can still follow along without hardware.

In [ ]:
SCOPE = 'OPENADC'
PLATFORM = 'CWHUSKY'

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
scope.bitbanger.i2c.set_defaults()
print(scope.bitbanger)

The above set some useful defaults, but we still need to define which Husky pins are connected to the target's clock and data lines.

In [ ]:
if scope._is_husky_plus:
    scope.bitbanger.data_pin = 'TIO1'
    scope.bitbanger.clock_pin = 'TIO2'
else:
    print('You must manually jumper TIO1 to USERIO_D0, and TIO2 to USERIO_D1.')
    scope.io.tio1 = 'high_z'
    scope.io.tio2 = 'high_z'
    scope.bitbanger.data_pin = 'USERIO_D0'
    scope.bitbanger.clock_pin = 'USERIO_D1'

We also disable the HS2 output clock since it is not needed and would only add noise:

In [ ]:
scope.io.hs2 = 'disabled'

As always, `scope.bitbanger` uses the ADC clock and divides it by `scope.bitbanger.clk_div` to drive bits out.

These settings give us the follow data rate:

In [ ]:
scope.bitbanger.clk_div = 52

print('bit time: %3.1f microseconds' % (1 / scope.clock.adc_freq * scope.bitbanger.clk_div * 1e6))
print('SCL clock speed: %4.1f KHz' % (scope.clock.adc_freq / scope.bitbanger.clk_div / 1e3))

Before talking to an I2C target, we must specific its 7-bit device address. `0x50` is for the 24CS32; use the correct address for your target.

In [ ]:
scope.bitbanger.i2c.device_address = 0x50

The 24CS32 datasheet discusses the need for a software reset to exit standby mode:

In [ ]:
from chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyBitBanger import BitBangerPacket

def swreset(scope):
    #              start 9 dummy clocks     START STOP
    pattern_data = [0,0, 1,1,1,1,1,1,1,1,1, 1,  0,0]
    clk_en       = [0,1, 1,1,1,1,1,1,1,1,1, 0,  0,1]
    pattern_en   = [1]*len(pattern_data)
    hiz          = [1]*len(pattern_data)
    trigger_en   = [0]*len(pattern_data)
    record_en    = [0]*len(pattern_data)
    
    packet = BitBangerPacket(pattern_data, hiz, pattern_en, record_en, trigger_en, clk_en)
    scope.bitbanger.sendpacket(packet)

In [ ]:
swreset(scope)

# Basic EEPROM Reads and Writes

Memory reads and writes are straightforward:

In [ ]:
scope.bitbanger.i2c.rread(0)

In [ ]:
scope.bitbanger.i2c.bwrite(0, 123)

In [ ]:
assert scope.bitbanger.i2c.rread(0) == 123

### Configuration Register

To access the 24CS32's configuration register, a different device address must be used (`0b101_1000 = 0x58`); the word address must be `0b1xxx_10xx_xxxx_xxxx` (e.g.: `0x8800`).

In [ ]:
scope.bitbanger.i2c.device_address

In [ ]:
scope.bitbanger.i2c.device_address = 0x58
scope.bitbanger.i2c.sread(0x8800, 2)

Let's now write the configuration register to turn on memory protection.

Addressing is like the read case. To change the configuration register we write 3 bytes; from MSB to LSB these are:
1. config register byte 0: `0bxxxx_xx_d9_d8`
   - D9: `EWPM` ("enhanced protection")
   - D8: `LOCK`
3. config register byte 1: `d7-d0`
   - bit x is `SWPx`
5. "confirmation byte": should match LOCK bit value from byte 0
   - `0b1001_1001 = 0x99` means LOCKED
   - `0b0110_0110 = 0x66` means UNLOCKED

In our example we won't play with the "lock" setting. We will turn on the enhanced protection and set SWP1 to write-protect the memory page starting at address `0x200`:

In [ ]:
def lock_page1(lock):
    scope.bitbanger.i2c.device_address = 0x58
    if lock:
        payload = [0x02, 0x02, 0x66] # set EWPM, turn on SWP1 (address page @ 0x200)
    else:
        payload = [0x02, 0x00, 0x66]
    scope.bitbanger.i2c.pwrite(0x8800, payload)
    scope.bitbanger.i2c.device_address = 0x50

With page 1 unlocked, we can write to 0x200:

In [ ]:
import time
import random

In [ ]:
lock_page1(False)
time.sleep(0.1)
wdata = random.randbytes(1)[0]
scope.bitbanger.i2c.bwrite(0x200, wdata)
time.sleep(0.1)
assert scope.bitbanger.i2c.rread(0x200) == wdata

With page 1 locked, writes still succeed but the memory data remains unchanged:

In [ ]:
lock_page1(True)
time.sleep(0.1)
scope.bitbanger.i2c.bwrite(0x200, wdata ^ 0xFF)
time.sleep(0.1)
assert scope.bitbanger.i2c.rread(0x200) == wdata

# Power Traces

In [ ]:
def capture_trace(sendcommand):
    scope.arm()
    sendcommand()
    ret = scope.capture()
    if ret:
        print("WARNING: Timeout happened during capture")
        return None
    wave = scope.get_last_trace()
    return wave

In [ ]:
scope.trigger.module = 'bitbanger'
scope.adc.samples = 110000
scope.adc.presamples = 1000
scope.gain.db = 13
scope.io.hs2 = 'disabled'

In [ ]:
lock_page1(False)

In [ ]:
trace = capture_trace(lambda:scope.bitbanger.i2c.bwrite(0, 0x55, trigger_bit=0))

In [ ]:
cw.plot(trace)

There are a **lot** of things jumping out of this power trace! It should look like this:

<img src='img/i2c_trace1.png' width=600>

First, we see that the target is very quiet before and after the operation.

Next, the power trace gets quite busy during the I2C transaction, which lasts this many ADC samples:

In [ ]:
scope.bitbanger.num_bits * scope.bitbanger.clk_div

(corresponding to samples 1000 to 3080)

We then see a few distinct phases of operations, with larger peaks and also very clear periodicity.

Let's look at the average of several traces:

In [ ]:
from tqdm.notebook import tnrange
import numpy as np

avgtrace = np.zeros(scope.adc.samples)
numtraces = 100
traces = []
for _ in tnrange(numtraces):
    trace = capture_trace(lambda:scope.bitbanger.i2c.bwrite(0, 0x55, trigger_bit=0))
    avgtrace += trace
    traces.append(trace)

avgtrace /= numtraces

In [ ]:
from bokeh.plotting import figure, show
from bokeh.resources import INLINE
from bokeh.io import output_notebook
from bokeh.models import Span, Legend, LegendItem
output_notebook(INLINE)
B = figure(width=1800, tools='pan, box_zoom, hover, reset, save')
xrange = list(range(scope.adc.samples))
t = B.line(xrange, trace, line_color='blue')
at = B.line(xrange, avgtrace - 0.4, line_color='green')
dt = B.line(xrange, abs(trace - avgtrace) - 0.8, line_color='red')

legend = Legend(items=[
    LegendItem(label='single trace', renderers=[t]),
    LegendItem(label='average trace', renderers=[at]),
    LegendItem(label='single - average', renderers=[dt]),
])

B.add_layout(legend)
show(B)

The traces should look like this:

<img src='img/i2c_trace2.png' width=1200>

The averaged trace still features the distinct markers, but they get weaker as time progresses.

One can also see periodic markers suggestive of a clock every 63 cycles; they are strong at the beginning but gradually weaken into noise over the first few thousand samples, whereas on the single trace they remain strong throughout.

What is clearly happening here is that the target has an internal clock that is not synchronized to Husky's ADC clock; this target clock is jittery and wanders over time, and so averaging several traces flattens it out.

**However**, the distinct markers sufficiently strong to remain clear.

To get a better sense of the target jitter, let's overlay several indidivual traces, focusing on the last marker near sample 102000:

In [ ]:
start = 101800
stop = 102200
from bokeh.palettes import inferno
from bokeh.plotting import figure, show
from bokeh.resources import INLINE
from bokeh.io import output_notebook
from bokeh.models import Span, Legend, LegendItem
import itertools
output_notebook(INLINE)
B = figure(width=1800)
xrange = list(range(stop-start))
numplots =  numtraces
colors = itertools.cycle(inferno(numplots))
for i in range(numplots):
    B.line(xrange, traces[i][start:stop], color=next(colors))
show(B)


We see a spread of approximately 80 samples.

For the earlier peak near sample 3950, jitter is only one sample:

In [ ]:
start = 3920
stop = 3950
B = figure(width=1800)
xrange = list(range(stop-start))
numplots =  numtraces
colors = itertools.cycle(inferno(numplots))
for i in range(numplots):
    B.line(xrange, traces[i][start:stop], color=next(colors))
show(B)


## Writing Locked vs Unlocked Memory

Let's now compare power traces of writes to locked vs unlocked memory and see what can be learned.

In [ ]:
lock_page1(False)

avgtrace_unlocked = np.zeros(scope.adc.samples)
traces = 200
for _ in tnrange(traces):
    trace_unlocked = capture_trace(lambda:scope.bitbanger.i2c.bwrite(0x200, 0x55, trigger_bit=0))
    avgtrace_unlocked += trace_unlocked

avgtrace_unlocked /= traces

In [ ]:
lock_page1(True)

avgtrace_locked = np.zeros(scope.adc.samples)
traces = 200
for _ in tnrange(traces):
    trace_locked = capture_trace(lambda:scope.bitbanger.i2c.bwrite(0x200, 0x55, trigger_bit=0))
    avgtrace_locked += trace_locked

avgtrace_locked /= traces

In [ ]:
B = figure(width=1800, tools='pan, box_zoom, hover, reset, save')
xrange = list(range(scope.adc.samples))
tu = B.line(xrange, avgtrace_unlocked, line_color='green')
tl = B.line(xrange, avgtrace_locked - 0.4, line_color='red')
dt = B.line(xrange, abs(avgtrace_locked - avgtrace_unlocked) - 0.8, line_color='blue')

legend = Legend(items=[
    LegendItem(label='unlocked average', renderers=[tu]),
    LegendItem(label='locked average', renderers=[tl]),
    LegendItem(label='difference', renderers=[dt]),
])

B.add_layout(legend)
show(B)

At first glance, the locked trace shows similar activty during the I2C transmission, then it's dead quiet.

But let's zoom into the I2C transmission, which you'll recall covers samples 1000 to 3080:

In [ ]:
samples = 3100
B = figure(width=1800, tools='pan, box_zoom, hover, reset, save')
xrange = list(range(samples))
tu = B.line(xrange, avgtrace_unlocked[:samples], line_color='green')
tl = B.line(xrange, avgtrace_locked[:samples], line_color='red')
dt = B.line(xrange, ((avgtrace_locked - avgtrace_unlocked)*10)[:samples] - 0.4, line_color='blue')

legend = Legend(items=[
    LegendItem(label='unlocked average', renderers=[tu]),
    LegendItem(label='locked average', renderers=[tl]),
    LegendItem(label='difference', renderers=[dt]),
])

B.add_layout(legend)
B.legend.location = 'bottom_left'
show(B)

There are a few clear differences during the I2C activity, and a particularly strong difference on the peak at sample 3029.

Let's grab and overlay the I2C line activity for context:

In [ ]:
scope.LA.enabled = True
scope.LA.clk_source = 'pll'
scope.LA.trigger_source = 'falling_tio1'
scope.LA.oversampling_factor = 4
scope.LA.downsample = 1
scope.LA.capture_group = 'CW 20-pin'
scope.LA.capture_depth = samples

In [ ]:
scope.LA.arm()
scope.bitbanger.i2c.bwrite(0x200, 0x55)

raw = scope.LA.read_capture_data()
io_data = scope.LA.extract(raw, 0)
io_clock = scope.LA.extract(raw, 1)

In [ ]:
shift_adjust = 0
shift = scope.adc.presamples - shift_adjust
io_data_shifted = [1]*shift
io_data_shifted.extend(io_data[:scope.adc.samples-shift])

io_clock_shifted = [1]*shift
io_clock_shifted.extend(io_clock[:scope.adc.samples-shift])

In [ ]:
from bokeh.models import Label
samples = 3100
B = figure(width=1800, tools='pan, box_zoom, hover, reset, save')
xrange = list(range(samples))
tu = B.line(xrange, avgtrace_unlocked[:samples], line_color='green')
tl = B.line(xrange, avgtrace_locked[:samples], line_color='red')
dt = B.line(xrange, ((avgtrace_locked - avgtrace_unlocked)*10)[:samples] - 0.4, line_color='blue')

sda = B.line(xrange, np.asarray(io_data_shifted[:samples])/10 + 0.2, line_color='green')
scl = B.line(xrange, np.asarray(io_clock_shifted[:samples])/10 + 0.4, line_color='brown')

label1 = Label(x=1000, y=0.1, text='device address (0x50)')
label2 = Label(x=1800, y=0.1, text='write address (0x02, 0x00)')

B.add_layout(label1)
B.add_layout(label2)

legend = Legend(items=[
    LegendItem(label='unlocked average', renderers=[tu]),
    LegendItem(label='locked average', renderers=[tl]),
    LegendItem(label='difference', renderers=[dt]),
    LegendItem(label='SDA', renderers=[sda]),
    LegendItem(label='SCL', renderers=[scl]),
])

B.add_layout(legend)
B.legend.location = 'bottom_left'
show(B)

If you don't have the hardware, here is what the traces should look like:

<img src='img/i2c_trace3.png' width=1200>

The power traces suggest that the target is behaving differently *during* the I2C write command, and at the very end of it, depending on whether or not the memory page is locked.

Whether this can be exploited to defeat the memory protection is left as an exercise to the reader.

Here lies one of the big advantages of `scope.bitbanger`: without its synchronized and perfectly repeatable bit-banging, it would be much harder to usefully acquire these types of power traces which provide potentially useful insight into the target's behaviour.